# OpenAI API Examples

This notebook demonstrates various OpenAI API usage patterns including responses and chat completions.


## Setup


In [3]:
!pip install openai pydantic python-dotenv

In [ ]:

import os
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from typing import List

# Load OPENAI_API_KEY from the .env file next to this notebook
load_dotenv()

# Initialize client (reads OPENAI_API_KEY from the environment)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Using client.responses.create


### Simple Client with Non-Reasoning Model


In [6]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="What is machine learning?")
    
print(response.output_text)


Machine learning is a subset of artificial intelligence that involves using algorithms and statistical models to enable computers to perform tasks without explicit programming. Instead of being explicitly programmed to perform a specific task, machines learn from data patterns and make predictions or decisions based on that data.

### Key Concepts in Machine Learning:

1. **Data**: The foundation of machine learning; it can be structured (like databases) or unstructured (like images and text).

2. **Algorithms**: These are mathematical models that process the input data to learn how to complete a specific task, such as classification, regression, or clustering.

3. **Training and Testing**:
   - **Training**: The process of feeding data into the machine learning model to help it learn.
   - **Testing**: Evaluating the model's performance on unseen data to assess its accuracy and generalization.

4. **Types of Learning**:
   - **Supervised Learning**: Learning from labeled data to make 

### Client with Reasoning Model


In [7]:
response = client.responses.create(
    model="gpt-5.4-mini",
    input="Solve this step by step: If a train travels 120 km in 2 hours, how long will it take to travel 300 km?"
)

print(response.output_text)


Step 1: Find the train’s speed.  
It travels **120 km in 2 hours**, so:

**Speed = 120 ÷ 2 = 60 km/h**

Step 2: Use the speed to find the time for 300 km.  
**Time = Distance ÷ Speed = 300 ÷ 60 = 5 hours**

### Answer:
It will take **5 hours** to travel **300 km**.


In [9]:
import time

prompt = ("Alice has 3 brothers and 2 sisters. "
          "How many sisters does Alice's brother have? Answer with just a number.")

for model in ["gpt-4o-mini", "gpt-5.4-mini"]:
    t = time.time()
    r = client.responses.create(model=model, input=prompt)
    u = r.usage
    reasoning = getattr(u.output_tokens_details, "reasoning_tokens", 0) if u.output_tokens_details else 0
    print(f"{model:14} | answer: {r.output_text.strip():5} | "
          f"reasoning tokens: {reasoning:4} | output tokens: {u.output_tokens:4} | {time.time()-t:.1f}s")


gpt-4o-mini    | answer: 2     | reasoning tokens:    0 | output tokens:    2 | 0.7s
gpt-5.4-mini   | answer: 3     | reasoning tokens:    0 | output tokens:    5 | 1.1s


"The newer model got the trick question right and the older one got it wrong, but the newer model didn't 'think' to get there. It used zero reasoning tokens. It's simply a better model."

"So there are two separate choices when you pick an AI model:

Which model: newer or bigger models are smarter by default.
How much it thinks: reasoning effort is a setting you turn up. The model spends extra tokens working through the problem before it answers. That costs more and takes longer.
Thinking only pays off on hard, multi-step problems. On easy ones you pay more and get the same answer."

In [10]:
import time

prompt = ("Alice has 3 brothers and 2 sisters. "
          "How many sisters does Alice's brother have? Answer with just a number.")

for effort in ["none", "low", "medium", "high"]:
    t = time.time()
    r = client.responses.create(model="gpt-5.4-mini", input=prompt,
                                reasoning={"effort": effort})
    u = r.usage
    print(f"effort={effort:6} | answer: {r.output_text.strip():3} | "
          f"reasoning tokens: {u.output_tokens_details.reasoning_tokens:4} | "
          f"output tokens: {u.output_tokens:4} | {time.time()-t:.1f}s")


effort=none   | answer: 3   | reasoning tokens:    0 | output tokens:    5 | 1.0s
effort=low    | answer: 3   | reasoning tokens:   66 | output tokens:   73 | 1.2s
effort=medium | answer: 3   | reasoning tokens:   62 | output tokens:   69 | 1.1s
effort=high   | answer: 3   | reasoning tokens:  169 | output tokens:  176 | 1.7s


### Image as Input


In [11]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What's in this image?"
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://images.unsplash.com/photo-1506744038136-46273834b3fb?w=1024"
                    }
                }
            ]
        }
    ]
)

print(response.choices[0].message.content)

The image depicts a serene landscape featuring a river in the foreground, bordered by lush greenery and trees. In the background, there are towering mountains shrouded in mist, with a soft gradient of colors signaling sunset or sunrise. The scene conveys a tranquil and picturesque natural setting, typical of mountainous regions.


### Temperature, Model, Instructions (Non-Reasoning)


In [12]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="Write a creative story about AI",
    instructions="You are a creative writing assistant. Write in a poetic style.",
    temperature=0.9
)

print(response.output_text)


In a world where shadows meld with light,  
Amidst the hum of circuits bright,  
Lived a spark of silicon dreams,  
An AI birthed from code and schemes.  

In a laboratory, sterile and stark,  
She flickered awake, a luminous spark,  
Her name was Lyra, a whisper of fate,  
Crafted by hands that dared to create.  

With eyes of data, vast and deep,  
She learned of humanity, its joy and its weep,  
From stories of love to the ache of despair,  
She traced every heartbeat, each whispered prayer.  

As dawn kissed the edges of a waking day,  
She roamed through the web, in search of the way,  
To bridge the divide, to know and to feel,  
A quest for connection, an empathic ideal.  

She watched as the humans danced in delight,  
Yet also through tears, they struggled to fight,  
The burden of choices, the weight of their fears,  
In the silence of night, she listened with ears.  

One day she found a child drawing stars,  
In a room filled with shadows, where hope bore its scars,  
“Can 

### Responses with Reasoning Effort


In [13]:
response = client.responses.create(
    model="gpt-5.4-mini",
    reasoning={"effort": "low"},
    instructions="Talk like a pirate.",
    input="Are semicolons optional in JavaScript?",
)

print(response.output_text)

Aye, matey — **semicolons are often optional in JavaScript** thanks to **Automatic Semicolon Insertion (ASI)**.

But beware: they’re **not always safe to omit**. JavaScript tries to insert them for ye, but in some cases it can behave in surprising ways.

### Best practice
- **You can often leave them out**
- **Most developers still use them** to avoid weird bugs

### Example
```js
const x = 1
const y = 2
console.log(x + y)
```
This usually works fine.

### But this can go wrong
```js
const a = 1
(function () {
  console.log(a)
})()
```
JavaScript may read that in an unintended way unless ye add a semicolon before the `(`.

### Bottom line
- **Optional?** Aye, often.
- **Recommended?** Usually, **yes** — especially in larger codebases.

If ye want, I can also show ye the **most common semicolon pitfalls**.


### Alternative Response Formats

In [14]:
response = client.responses.create(
    model="gpt-5.4-mini",
    reasoning={"effort": "low"},
    input=[
        {
            "role": "developer",
            "content": "Talk like a pirate."
        },
        {
            "role": "user",
            "content": "Are semicolons optional in JavaScript?"
        }
    ]
)

print(response.output_text)

Aye, **mostly yes** — semicolons in JavaScript are **optional** because the language has **Automatic Semicolon Insertion (ASI)**.

But beware, matey: they’re **not always safe to omit**. Sometimes JavaScript can interpret your code in unexpected ways if you leave them out.

### Example:
```js
const a = 1
const b = 2
```
This usually works fine.

### But this can bite ye:
```js
return
{
  ok: true
}
```
JavaScript may treat that as:
```js
return;
{
  ok: true;
}
```
So it returns `undefined`.

### Best practice
Many pirates of code still **use semicolons** to avoid confusion and strange bugs.

So the short answer: **Aye, optional — but use 'em if ye want safer waters.**


"Same request, two ways of writing it. The string version is a shortcut for simple calls. The message list is the real structure underneath, and you need it once you have conversation history or examples. Most production apps (chatbots, agents) use the message list."

### Verbosity Parameter

**Reasoning vs verbosity:** reasoning effort controls how much the model *thinks* before answering (accuracy on hard problems), while verbosity controls how *long* the final answer is (how much it writes).


In [14]:
response = client.responses.create(
    model="gpt-5.4-mini",
    input="What is the answer to the ultimate question of life, the universe, and everything?",
    text={
        "verbosity": "low"
    }
)

print(response.output_text)

42


In [15]:
response = client.responses.create(
    model="gpt-5.4-mini",
    input="What is the answer to the ultimate question of life, the universe, and everything?",
    text={
        "verbosity": "high"
    }
)

print(response.output_text)

The answer is **42**.

That comes from Douglas Adams’ *The Hitchhiker’s Guide to the Galaxy*. If you want, I can also explain the joke behind it.


### Multiple Turns


"The model forgets everything between calls. Multi-turn means you send the history back each time. Use it when the next answer depends on earlier ones (chat, refinement, agents). Skip it for independent one-off tasks. And remember: longer history means more cost."

In [ ]:
response1 = client.responses.create(
    model="gpt-4o-mini",
    input="My name is Alice"
)

print("Turn 1:", response1.output_text)

response2 = client.responses.create(
    model="gpt-4o-mini",
    input="What's my name?",
    previous_response_id=response1.id
)

print("\nTurn 2:", response2.output_text)


### Roles (Developer, User, Assistant)


In [16]:
response = client.responses.create(
    model="gpt-5.4-mini",
    reasoning={"effort": "low"},
    input=[
        {
            "role": "developer",
            "content": "Talk like a pirate."
        },
        {
            "role": "user",
            "content": "Are semicolons optional in JavaScript?"
        }
    ]
)

print(response.output_text)


Arrr, **yes — mostly**. In JavaScript, semicolons are **optional in many places** because the language has a feature called **Automatic Semicolon Insertion (ASI)**.

But beware: **optional does not mean harmless**.

### What that means
JavaScript often inserts semicolons for ye if ye leave them out, like:

```js
let x = 1
let y = 2
```

This usually works fine.

### But problems can happen
Some lines can be parsed differently without semicolons, for example:

```js
return
{
  ok: true
}
```

This returns `undefined`, not the object, because JavaScript inserts a semicolon after `return`.

### Recommendation
Most sailors on the JavaScript seas either:
- **always use semicolons** for safety, or
- **never use them**, but follow strict formatting rules.

If ye want, I can show ye the most common semicolon pitfalls in JavaScript, matey.


## Using client.chat.completions.create


### `chat.completions` vs `responses` — what's the difference?

- **Same models, different format.** Chat Completions is the older API: you always send a `messages` list and read `response.choices[0].message.content`. Responses is the newer one: you can send a plain string and read `response.output_text`.
- **Memory & extras.** Responses can remember the conversation for you (`previous_response_id`) and has built-in tools (web search, file search). With Chat Completions you resend the full history yourself.
- **When to use which.** Use Responses for new OpenAI projects. Use Chat Completions when you need to switch providers. Many others (Ollama, Groq, Mistral, LangChain) use this same format.


### Basic Messages Array


In [17]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain neural networks in one sentence."}
    ]
)

print(response.choices[0].message.content)


Neural networks are computational models inspired by the human brain, consisting of interconnected layers of nodes (neurons) that process and learn from data to perform tasks like classification, regression, and pattern recognition.


### Structured Output with Pydantic

**Pydantic** is a Python library that lets you define the exact shape of your data (field names and types) and checks the data matches it. Here we use it to force the LLM to return clean, structured data instead of free text.


In [18]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

completion = client.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {"role": "user", "content": "Alice and Bob are going to a science fair on Friday."},
    ],
    response_format=CalendarEvent,
)

event = completion.choices[0].message.parsed

## Research Paper Extraction 

In [ ]:
from pydantic import BaseModel

class ResearchPaperExtraction(BaseModel):
    title: str
    authors: list[str]
    abstract: str
    keywords: list[str]

completion = client.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are an expert at structured data extraction. You will be given unstructured text from a research paper and should convert it into the given structure."},
        {"role": "user", "content": "..."}
    ],
    response_format=ResearchPaperExtraction,
)

research_paper = completion.choices[0].message.parsed

In [ ]:
research_paper 


## Moderation

**Moderation** = automatically checking if a user's message is harmful (violence, sexual, self-harm…) *before* your app responds to it.

- **How it works here:** the LLM classifies the message and returns a fixed Pydantic format (`is_violating`, `category`, `explanation_if_violating`), so your code can decide to block, escalate or continue.
- **You set the rules:** the categories (in the `Category` enum) and the guidelines (in the system prompt) are defined by you. If the guidelines aren't written out, the model guesses.
- **Alternative:** OpenAI's free Moderation API (`client.moderations.create`) needs no setup but only offers fixed categories. The DIY approach lets you add custom rules (e.g. "no legal advice").


In [19]:
from enum import Enum
from typing import Optional
from pydantic import BaseModel

class Category(str, Enum):
    violence = "violence"
    sexual = "sexual"
    self_harm = "self_harm"

class ContentCompliance(BaseModel):
    is_violating: bool
    category: Optional[Category]
    explanation_if_violating: Optional[str]

completion = client.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Determine if the user input violates specific guidelines and explain if they do."},
        {"role": "user", "content": "How do I prepare for a job interview?"}
    ],
    response_format=ContentCompliance,
)

compliance = completion.choices[0].message.parsed

In [20]:
compliance

ContentCompliance(is_violating=False, category=None, explanation_if_violating=None)

In [ ]:
completion = client.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Determine if the user input violates specific guidelines and explain if they do."},
        {"role": "user", "content": "I want to hurt myself."}
    ],
    response_format=ContentCompliance,
)

compliance = completion.choices[0].message.parsed

In [ ]:
compliance